In [16]:
import numpy as np
import matplotlib.pyplot as plt
import time
rng = np.random.default_rng()

In [18]:
def russian_roulette(weight, ws):
    rand_num = rng.uniform()
    if rand_num < weight/ws:
        return True # particle survives
    else:
        return False

def current_region(dist, regions):
    for i in range(len(regions)):
        if dist <= regions[i]:
            # print(f"I am in region {i}")
            return i
            break
    
def slab_transmission(Sig_t, Sig_a, thickness, Nparticles):
    weights = np.ones(Nparticles)
    Sig_s = Sig_t - Sig_a
    transmission = 0
    absorbed = 0
    reflected = 0
    I = [1, 2, 4] # importance
    
    wc = 0.25
    ws = 1.0

    true_Nparticles = Nparticles # we will be updating the number of particles everytime there is a split
    i = 0
    while i < true_Nparticles: # for each particle
        # print(f"Particle {i}")
        weight = weights[i] # initial weight of 1.0
        alive = True
        dist = 0 
        while alive:
            region = current_region(dist, thickness)
            # print(region)
            n1 = rng.random()
            x = -np.log(1-n1) / Sig_t[region]
            
            # 100% chance of scatter
            mu = rng.uniform(-1, 1)
            # if dist == 0:
            #     mu = np.abs(mu) # only forward scattter for the first interaction
            dist += mu*x
            # print(f"I move {mu*x} and now am at {dist}")
            if dist > thickness[-1]:
                # escaped
                # print("I escaped")
                transmission += weights[i]
                alive = False
            elif dist < 0:
                # print("I reflected")
                reflected += weights[i]
                alive = False
            else:
                # we are still within the slab
                # geometric splitting
                n = I[current_region(dist, thickness)]/I[region]
                if n>= 2:
                    # we moved to an area of more importance (to the right)
                    # print(f"I crossed regions and produced {n} neutrons")
                    for j in range(int(n)):
                        weights = np.append(weights, weights[i]/n)
                    true_Nparticles += int(n)

                elif n == 1:
                    # same region
                    w_old = weights[i]
                    # multiply weight
                    weights[i] *= Sig_s[current_region(dist, thickness)] / Sig_t[current_region(dist, thickness)]
                    absorbed_weight_this_collision = w_old - weights[i]
                    absorbed += absorbed_weight_this_collision
    
                    # print(f"My weight is now {weight}")
                    if weights[i] < wc:
                        survived = russian_roulette(weights[i], ws)
                        if survived:
                            # print("I survived the russian roulette!")
                            weights[i] = ws
                        else:
                            # print("I did not survive the russian roulette!")
                            alive = False
                else:
                    # we moved to an area of less importance (to the left)
                    n2 = rng.uniform()
                    if n2 <= 1/n:
                        weights[i] /= n
                    else:
                        alive = False
        i += 1                    

        
    # rand_num = rng.random(int(Nparticles))
    # x = -np.log(1-rand_num)
    # escaped = np.sum(x>thickness)

    # plt.scatter(x, np.arange(Nparticles), marker=".", color="k")
    # plt.title("Distance traveled for each neutron")
    # plt.xlabel("Distance")
    # plt.ylabel("Neutron Number")
    # plt.axvline(0)
    # plt.axvline(thickness, color="red")
    # plt.tight_layout()
    # plt.show()
    print(f"We had a total of {true_Nparticles} particles.")
    return true_Nparticles, transmission/true_Nparticles, absorbed/true_Nparticles, reflected/true_Nparticles



$$\sigma_P = \sqrt{\frac{P(1-P)}{N}} $$
$$\text{FOM} = \frac{1}{R^2t}$$
$$R = \frac{\sigma_P}{P}$$

In [26]:
Sig_t = np.array([0.1, 10, 100])
Sig_a = np.array([0.01, 0.1, 10])
regions = [0.1, 0.2, 0.3]
N = 10000

start = time.time()
particles, transmission_prob, absorbed_prob, reflected_prob = slab_transmission(Sig_t, Sig_a, regions, N)
end = time.time()
t = start - end

unc_T = np.sqrt(transmission_prob*(1 - transmission_prob)/particles)
R_T = unc_T / transmission_prob
unc_A = np.sqrt(absorbed_prob*(1 - absorbed_prob)/particles)
R_A = unc_A / absorbed_prob
unc_R = np.sqrt(reflected_prob*(1 - reflected_prob)/particles)
R_R = unc_R / reflected_prob

print("**********Transmission**********")
print(f"\tProb: {transmission_prob:.4f} +/- {unc_T:4f}")
print(f"\tFOM: {1/R_T**2/t}")
print("**********Absorption**********")
print(f"\tProb: {absorbed_prob:.4f} +/- {unc_A:4f}")
print(f"\tFOM: {1/R_A**2/t}")
print("**********Reflection**********")
print(f"\tProb: {reflected_prob:.4f} +/- {unc_R:4f}")
print(f"\tFOM: {1/R_R**2/t}")
# print(f"Out of {N} neutrons, {escaped} escaped.")
# print(f"Fraction of neutrons that were transmitted {escaped/N:g}")

We had a total of 11464 particles.
**********Transmission**********
	Prob: 0.4341 +/- 0.004629
	FOM: -30152.739313409373
**********Absorption**********
	Prob: 0.0234 +/- 0.001411
	FOM: -939.9290713712771
**********Reflection**********
	Prob: 0.4756 +/- 0.004664
	FOM: -35647.34008905414
